In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm
#from passwords import *
import pickle
import warnings
warnings.filterwarnings("ignore")

### Functions

In [2]:
# # download from s3
# def download_from_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_path, str_project):
#     # init client
#     cls_client = boto3.client(
#         's3',
#         aws_access_key_id=aws_access_key_id,
#         aws_secret_access_key=aws_secret_access_key,
#     )
#     # download file
#     cls_client.download_file(
#         str_project, 
#         str_bucket_path, 
#         str_local_path,
#     )

In [3]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # download file
    cls_client.download_file(
        str_project, 
        str_bucket_path, 
        str_local_path,
    )

In [4]:
# # upload to s3
# def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
#     # init client
#     cls_client = boto3.client(
#         's3',
#         aws_access_key_id=aws_access_key_id,
#         aws_secret_access_key=aws_secret_access_key,
#         aws_session_token=None,
#     )
#     # upload
#     cls_client.upload_file(
#         str_local_path, 
#         str_bucket_name, 
#         str_bucket_key,
#     )

In [5]:
# upload to s3
def upload_to_s3(str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [6]:
str_project = '20231010-gen-xii'
str_variant = 'noPTImodel10'

### Get the nonleaky features

In [7]:
list_str_model = [
    '01_ad',
    '02_pricing_pd',
    '03_pricing_lgd',
]
list_cols_all = ['bigaccountid__app']
for str_model in tqdm(list_str_model):
    str_filename = 'df_non_leaky.csv'
    str_local_path = f'../../../{str_model}/01_data_prep/05_leaky_features/04_write_dfs/output/{str_filename}'
    list_cols = list(pd.read_csv(str_local_path)['feature'])
    list_cols_all += list_cols
list_cols_all = list(dict.fromkeys(list_cols_all))
print(f'There are {len(list_cols_all)} non-leaky features')

100%|██████████| 3/3 [00:00<00:00, 22.65it/s]

There are 1585 non-leaky features


### Import PD data

In [8]:
list_str_df = [
    'train',
    'valid',
    'test',
]
list_df = []
for str_df in tqdm(list_str_df):
    # download
    str_filename = f'df_{str_df}_raw.gzip'
    str_bucket_path = f'02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
#         aws_access_key_id=AWS_ACCESS_KEY_ID, 
#         aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
    # read
    df = pd.read_parquet(str_local_path)
    list_cols = [col for col in list_cols_all if col in list(df.columns)]
    df = df[list_cols].copy()
    # make col
    df['data_set'] = str_df
    # rm
    os.remove(str_local_path)
    # append
    list_df.append(df)
# concatenate
df = pd.concat(list_df)

# show
df

100%|██████████| 3/3 [00:08<00:00,  2.77s/it]


,bigaccountid__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf105__tu,linkf195__tu,linkf193__tu,linkb012__tu,linkf032__tu,...,payment__app,bookvalue__app,miles_odometer__app,bitnew__app,vehicleyear__app,amtfinanced__app,vehiclemake__app,inttermbump__app,linkf189__tu,data_set
38325,1337511,0.0,0.0,0.0,152.0,40.0,1141.0,40.0,0.0,N,...,398.83,10550.0,46429.0,False,2009,14532.90,Chevrolet,6.0,40.0,train
38324,1337511,0.0,0.0,0.0,152.0,40.0,1141.0,40.0,0.0,N,...,398.83,10550.0,46429.0,False,2009,14532.90,Chevrolet,6.0,40.0,train
38326,1337528,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,420.66,11525.0,58628.0,False,2011,18439.47,Hyundai,6.0,NaN,train
38327,1337528,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,420.66,11525.0,58628.0,False,2011,18439.47,Hyundai,6.0,NaN,train
38329,1337539,NaN,0.0,3078.0,5.0,5.0,1802.0,5.0,0.0,N,...,440.04,14025.0,29855.0,False,2012,18857.27,Nissan,0.0,769.0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,4812498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,509.16,13750.0,22961.0,False,2017,21007.00,Nissan,0.0,NaN,test
7863,4812498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,None,...,509.16,13750.0,22961.0,False,2017,21007.00,Nissan,0.0,NaN,test
20360,4812503,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,N,...,492.00,19718.0,21.0,True,2020,20545.21,Hyundai,0.0,NaN,test
20361,4812503,0.0,0.0,0.0,1182.0,1182.0,1182.0,1182.0,0.0,N,...,492.00,19718.0,21.0,True,2020,20545.21,Hyundai,0.0,NaN,test


### Get preprocessor

In [9]:
list_str_filename = [
    'preprocessing.py',
    'cls_model_preprocessing.pkl',
]
for str_filename in tqdm(list_str_filename):
    # download
    str_bucket_path = f'01_ad/02_model/{str_variant}/00_preprocessing/01_create_preprocessor/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
#         aws_access_key_id=AWS_ACCESS_KEY_ID, 
#         aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
# import
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))
# rm
os.remove(str_local_path)

100%|██████████| 2/2 [00:00<00:00, 12.19it/s]


### Preprocess data

In [10]:
%%time

df = cls_model_preprocessing.transform(df)

# show
df

NaN Replacer: 5.4704 sec.


100%|██████████| 3/3 [00:00<00:00, 33.17it/s]

Unable to convert vehiclemodel__app to string, not found in data
Set strings: 0.092592 sec.


Boolean Replacer: 3.7271 sec.


100%|██████████| 1571/1571 [00:01<00:00, 803.74it/s]


...


100%|██████████| 1576/1576 [00:02<00:00, 588.97it/s]


Replace inf and -inf with NaN: 2.9447 sec.
Imputer: 2.157 sec.
Map term: 0.095829 sec.
Map PTI: 0.10712 sec.


100%|██████████| 9/9 [00:00<00:00, 858.96it/s]

Round values: 0.013403 sec.
Preprocessing Model: 23.432 sec.
CPU times: user 14.8 s, sys: 8.59 s, total: 23.4 s
Wall time: 23.4 s


,bigaccountid__app,linkf060__tu,linkf045__tu,linkf079__tu,linkf185__tu,linkf105__tu,linkf195__tu,linkf193__tu,linkb012__tu,linkf032__tu,...,linkf189__tu,data_set,year,factor,ENG-applicationdate__app_month,ENG-applicationdate__app_quarter,ENG-payment_to_income,ENG-loan_to_value,ENG-vehicle_age,ENG-dealership_age
38325,1337511.0,0.0,0.0,0.000000,152.0,40.0,1141.0,40.0,0.0,n,...,40.0,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932
38324,1337511.0,0.0,0.0,0.000000,152.0,40.0,1141.0,40.0,0.0,n,...,40.0,train,2013,1.256262,10,4,0.09,1.370370,4.0,1.284932
38326,1337528.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,nan,...,0.0,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863
38327,1337528.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,nan,...,0.0,train,2013,1.256262,10,4,0.15,1.586207,2.0,1.569863
38329,1337539.0,0.0,0.0,3866.774083,5.0,5.0,1802.0,5.0,0.0,n,...,769.0,train,2013,1.256262,10,4,0.03,1.342857,1.0,3.961644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,4812498.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,nan,...,0.0,test,2019,1.144717,12,4,0.09,1.548387,2.0,10.010959
7863,4812498.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,nan,...,0.0,test,2019,1.144717,12,4,0.09,1.548387,2.0,10.010959
20360,4812503.0,0.0,0.0,0.000000,1182.0,1182.0,1182.0,1182.0,0.0,n,...,0.0,test,2019,1.144717,12,4,0.12,1.044444,-1.0,0.208219
20361,4812503.0,0.0,0.0,0.000000,1182.0,1182.0,1182.0,1182.0,0.0,n,...,0.0,test,2019,1.144717,12,4,0.12,1.044444,-1.0,0.208219


### Get models

In [11]:
list_str_model = [
    '01_ad',
    '02_pricing_pd',
    '03_pricing_lgd',
]
dict_models = {}
for str_model in tqdm(list_str_model):
    # download model
    str_filename = f'final_model.pkl'
    str_bucket_path = f'{str_model}/02_model/{str_variant}/03_final_model/{str_filename}'
    str_local_path = f'./{str_filename}'
    download_from_s3(
#         aws_access_key_id=AWS_ACCESS_KEY_ID, 
#         aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )
    # import model
    cls_model_inference = pickle.load(open(str_local_path, 'rb'))['model_inference']
    # rm
    os.remove(str_local_path)
    # assign to dictionary
    dict_models[str_model] = cls_model_inference

100%|██████████| 3/3 [00:00<00:00,  3.64it/s]


### Generate predictions

In [12]:
list_str_colname = ['uniqueid']
for str_model, cls_model_inference in dict_models.items():
    print(f'Generating predictions for {str_model}...')
    # get cols in model
    list_cols_model = list(cls_model_inference.feature_names_)
    # create col name
    str_colname = f'yhat_{str_model[3:]}'
    # append
    list_str_colname.append(str_colname)
    # predict
    if 'lgd' in str_model:
        df[str_colname] = cls_model_inference.predict(df[list_cols_model])
    else:
        df[str_colname] = cls_model_inference.predict_proba(df[list_cols_model])[:,1]
# show
df[list_str_colname]

Generating predictions for 01_ad...
Generating predictions for 02_pricing_pd...
Generating predictions for 03_pricing_lgd...


,uniqueid,yhat_ad,yhat_pricing_pd,yhat_pricing_lgd
38325,1.337511e+14,0.654475,0.465386,0.670604
38324,1.337511e+14,0.663566,0.493285,0.670604
38326,1.337528e+14,0.257685,0.214348,0.626033
38327,1.337528e+14,0.237884,0.199132,0.626033
38329,1.337539e+14,0.550738,0.531353,0.628371
...,...,...,...,...
7864,4.812499e+14,0.336901,0.145701,0.592475
7863,4.812499e+14,0.343272,0.153010,0.592475
20360,4.812504e+14,0.341951,0.118688,0.689688
20361,4.812504e+14,0.332062,0.104094,0.689688


### Upload data to s3

In [13]:
%%time

# convert non-numeric to string
for col in tqdm(df.columns):
    if df[col].dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)

# write locally
str_filename = 'df_pd_pre_with_yhats.gzip'
str_local_path = f'./{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

# upload to s3
str_bucket_key = f'ad_hoc/get_predictions/{str_variant}/{str_filename}'
upload_to_s3(
#     aws_access_key_id=AWS_ACCESS_KEY_ID, 
#     aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=str_bucket_key, 
    str_bucket_name=str_project,
)
# rm
os.remove(str_local_path)

100%|██████████| 1592/1592 [00:00<00:00, 1926.86it/s]


CPU times: user 40.7 s, sys: 864 ms, total: 41.6 s
Wall time: 39.5 s


### Clean-up

In [14]:
list_str_filename = [
    'preprocessing.py',
]
for str_filename in list_str_filename:
    os.remove(str_filename)